In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab09.ipynb")

<img src="data6.png" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

# Lab 9 – APIs and Prompt Engineering

## Data 6, Summer 2025

In this lab, you will explore how to interact with large language models (LLMs) using the [OpenAI API](https://openai.com/api/). You’ll learn how to structure prompts, understand the roles used in chat messages, and analyze how different instructions affect the model’s response. The lab also introduces the concept of a “chain of command” that determines how the model prioritizes instructions from system, developer, and user roles.

This lab is due on **Wednesday, at 11:00 PM**. You must submit the assignment to Gradescope.

In [ ]:
from datascience import *
import numpy as np
from tqdm import tqdm
import json
from IPython.display import YouTubeVideo, HTML, display
from ipywidgets import interact, widgets
%matplotlib inline
%pip install openai # You may see an Error pop up, that's fine.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


# Section 1: Working with APIs

An **Application Programming Interface (API)** allows programmers (people who write code) to access code that other people have written. One large benefit of using APIs is that you don't have to understand how a function is implemented, but just its functionality.

An example is your usage of the `Numpy` library. When you call `np.average`, you can trust that the function will return the mean value of all the values in the `Numpy` array that was passed in, but you don't need to know exactly how the function is implemented!

Most likely, you've used [chatgpt.com](https://chatgpt.com/) in order to access a Large Language Models (LLMs). But now, with our technical skills, we can use an API request to access these LLMs! 

<hr style="border: 1px solid #fdb515;" />

# Question 1 - Set your API Key

To start off, we've shared your API key with you through email. This API key allows Open AI to recognize who is using their API, and charge us accordingly. Set your `API_KEY` to be a string of alphanumeric characters that we provide you. 

**IMPORTANT NOTE: Please DO NOT share your API key outside of this class. We will disable the API keys after the semester has ended, so if you'd like to play around with your code after the term, you'll have to get your own API key.**

In [ ]:
"""
When writing code that will go to production (e.g. be seen by other people), 
you'll usually set your API key as an environment variable via your terminal.
"""
# Find the API key we've given you in EdStem
API_KEY = ...

In [ ]:
grader.check("q1")

<hr style="border: 1px solid #fdb515;" />

# Question 2 – Understanding the API Call
## API Request Structure

Let's start by taking a look at the [chat completion](https://platform.openai.com/docs/api-reference/chat/create) functionality of these Open AI models. As you can see, you can think of an API request as just a fancy function call.

<!-- BEGIN QUESTION -->

---

## Question 2a
When making a chat completion request, what is the data type of `messages` (e.g. integer, boolean, etc)?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

## Question 2b
When making a chat completion request, what is the data type of each element within `messages` (e.g. integer, boolean, etc)?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 1px solid #fdb515;" />

# Question 3 – API Request Return Structure

At its heart, an API request to the OpenAI API just requires some choice of LLM to be supplied as an argument, as well as the message we want to send to the LLM. The model is a String, and the message is a list of dictionaries. Each dictionary contains a mapping of "role" to the corresponding role, and "content" to the actual message. Let's start by making our first API request:

In [ ]:
from openai import OpenAI
client = OpenAI(api_key = API_KEY)

In [ ]:
message = [
    {"role": "user", "content": "Hi there!"} # We create a list of dictionaries containing the prompt.
]

completion = client.chat.completions.create(
  model="gpt-4o-mini", # Picking the model 
  messages=message # Supplying the message
)

In [ ]:
type(completion) # Run this cell

Notice that a [chat completion object](https://platform.openai.com/docs/api-reference/chat/object) is returned from the API call. 

We can then convert this into json so we can parse the outputs! Alternatively, the API reference also shows you how to access it via **dot notation**, which is not in scope for this course.

In [ ]:
response_json = completion.model_dump_json() 
response_dict = json.loads(response_json)
type(response_dict) # Notice that this is a dictionary!

Super cool! We turned a chat completion object into a dictionary, which we know how to parse. Let's take a look inside the dictionary...

In [ ]:
response_dict

Given `response_dict`, assign `hello_llm` to the expression that would evaluate to the response from the LLM (in this case, "Hello! How can I assist you today?").

In [ ]:
hello_llm = ...
hello_llm 

In [ ]:
grader.check("q3")

<hr style="border: 1px solid #fdb515;" />

# Question 4 – Making Your Own API Request!

Try modifying and making your own API request to see what the best restaurant in Berkeley is.

In [ ]:
# The system content allows us to specify external instructions to the model.
message2 = [{"role": "user", "content": ...}]
completion = client.chat.completions.create(
  model="gpt-4o-mini", # Picking model 
  messages=message2 # Supplying the message
)
print(completion.choices[0].message.content) # Accesses the response from the model using dot notation

In [ ]:
grader.check("q4")

<hr style="border: 1px solid #fdb515;" />

# Question 5 – Creating A Helper Function

Since writing the same lines of code over and over can be cumbersome, let's write a utility function to help us make these API requests and process them.

Fill in `prompt`, which takes in a String representing the model we want to use, as well as a list of dictionaries representing the message we want to send to the model. This function will return the message the LLM sends back to us, since we don't care about the other parts of the chat completion object.

**_Note_:** Do not use the dot notation method!

In [ ]:
def prompt(model, message):
    """
    Makes an API request to a LLM and returns the message.

    Inputs:
    model: A String representing the model we're using
    message: A list of dictionaries, where each dictionary has two entries.
    The first entry in each dictionary should have the key "role", and a value representing the role.
    The second entry in each dictionary should have the key "content", and a value representing the content.

    Returns:
    A String representing the result of an API request to the model, parsed to just get the message back.

    >>> model_choice = "gpt-4o-mini"
    >>> message = [{"role": "user", "content": "Hi there!"}]
    >>> isinstance(prompt(model_choice, message), str)
    True
    """
    ...

In [ ]:
# Test your implementation of prompt here
model_choice = "gpt-4o-mini"
message = [{"role": "user", "content": "Hi there!"}]
response = prompt(model_choice, message)
response

In [ ]:
grader.check("q5")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Section 2 – Chain of Command and Prompt Engineering

You may have noticed that in the `message` list:

```python
message = [{"role": "user", "content": "Hi there!"}]
```

The dictionary contains a `"role"` key with the `"user"` value. This defines who is "speaking" to the model. 

According to [OpenAI’s Chain of Command Spec (2025)](https://model-spec.openai.com/2025-02-12.html#chain_of_command), different roles have different levels of authority over the assistant’s behavior. They are designed to **follow all applicable instructions,** but only when they don’t conflict with **higher-authority instructions** or newer instructions at the same authority level. Below is a table of the authority levels that OpenAI models use: 

| **Authority Level** | **Source**                                                                 |
|---------------------|-----------------------------------------------------------------------------|
| **Platform**        | System messages and [Model Spec](https://model-spec.openai.com) "platform" sections |
| **Developer**       | Developer messages and developer-facing sections of the Model Spec         |
| **User**            | User messages and user-facing instructions                                 |
| **Guideline**       | Lower-priority best practices, general behavior suggestions                 |
| **No Authority**    | Assistant messages, tool outputs, untrusted/quoted data                    |

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

# Question 6 – Why Does the Chain of Command Matter?

What could go wrong if a language model always followed what the user or developer said, even when it goes against platform rules?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

# Question 7 – Various Roles

In addition to the `"user"` role, there are other roles that can be included in a `message` prompt to help shape the model’s behavior. Below are such roles available to the public:

| **Role**    | **Authority Level** | **Purpose**                                                                        |
| ----------- | ------------------- | ---------------------------------------------------------------------------------- |
| `system`    | **Platform**        | Sets the assistant’s core behavior, rules, and safety boundaries.                  |
| `developer` | **Developer**       | Adds tone, formatting, and style instructions.                                     |
| `user`      | **User**            | Represents the person prompting the model with a question or command.              |
| `assistant` | **No Authority**    | Represents previous replies from the model, used to maintain conversation history. |
| `tool`      | **No Authority**    | Used when the model calls an external tool (e.g., function calling).               |

Below is an example message using the various roles.

In [ ]:
message3 = [{
        "role": "system",
        "content": "You are a neutral and fact-based assistant. Do not make up answers or express opinions."
    },
    {
        "role": "developer",
        "content": "Respond in a formal tone, no emojis, and be brief."
    },
    {
        "role": "user",
        "content": "Why is it important for a model to refuse to answer certain questions?"
    }
]

response3 = prompt(model_choice, message3)
response3

In real life, who might be writing the messages for the system, developer, and user roles? What is each person trying to do?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

# Question 8 – Your Turn for Prompt Engineering!

Now that you have better understanding of the chain of command and various roles, it's your turn to create `message4` that uses `"system"`, `"developer"`, and `"user"` roles. Except, let's make sure that our prompt has a bit more structure to it as well. 

1. Follow the [LLM prompting guidelines](https://direct.mit.edu/view-large/figure/4722326/coli_a_00502_i004.tif) to make your prompt better!
2. Then, read over the [Open AI guide](https://platform.openai.com/docs/guides/prompt-engineering#tactic-provide-examples) on how to do few-shot prompting.

Your updated prompt should:
1. Include a brief context (15 words or less) 
2. Enumerate options as alphabetical multiple choice, separated by a new line. For this, you can include the newline character "\n" in your prompt.
3. Specify constraints
4. Specify actions under uncertainty
5. Provide 2 examples of expected inputs and outputs

**_Note:_** The two references seem to disagree as to whether the context should be provided at the very end (as the Open AI guide suggests), or at the beginning (as the MIT paper suggests). The choice is up to you!

In [ ]:
message4 = ...
response4 = prompt(model_choice, message4)
response4

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<hr style="border: 1px solid #fdb515;" />

# Question 9 – Exploring Different Models

So far, we have been using the `gpt-4o-mini` model, but OpenAI offers many others. You can view them on the [OpenAI pricing page](https://platform.openai.com/docs/pricing).

For this question, play around with earlier messages (such as `message`, `message2`, etc.) or a new message and try running it with one of the following models `gpt-3.5-turbo`, `gpt-4o`, or `gpt-4.1`.

In [ ]:
prompt("gpt-4.1", message3)

What do you notice about the differences in the response? Does the **release date** of the model seem to affect how up-to-date or detailed the response is? What about how fast the model takes to respond?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Pets of Data 6

You too can be a hero! **Hero** just wanted to let you know!!!

<img src="hero.jpeg" width="50%" alt="Cute dog names hero"/>

## Submission

For this assignment, you will submit only the PDF. You can download the notebook as a PDF by clicking on `File -> Save and Export Notebook As... -> Webpdf` or `File -> Save and Export Notebook As... -> PDF`. 

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()